In [6]:
import os
os.makedirs("RandomLASSO_Coefs", exist_ok=True)
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lars, LassoLars
from sklearn.metrics import f1_score, average_precision_score, mean_squared_error
from joblib import Parallel, delayed

# Suppressing convergence warnings
# Expected behavior during grid search in extreme high-dimensional (p >> n) genomic space.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ---------------------------------------------------------
# Math Helpers
# ---------------------------------------------------------
def calc_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calc_rme(beta_true, beta_hat):
    return np.linalg.norm(beta_hat - beta_true) / (np.linalg.norm(beta_true) + 1e-10)

def calc_rme_nonzeros(beta_true, beta_hat):
    mask = beta_true != 0
    return np.linalg.norm(beta_hat[mask] - beta_true[mask]) / (np.linalg.norm(beta_true[mask]) + 1e-10)

# ---------------------------------------------------------
# Random LASSO Engines (Rebranded)
# ---------------------------------------------------------
def generate_importance_scores(b, X_train_scaled, y_train_scaled, q1, p, alpha_val):
    rs = np.random.default_rng(b + 100)
    n = X_train_scaled.shape[0]
    boot_rows = rs.choice(n, size=n, replace=True)
    boot_cols = rs.choice(p, size=q1, replace=False)
    
    model = LassoLars(alpha=alpha_val, max_iter=2000)
    model.fit(X_train_scaled[boot_rows][:, boot_cols], y_train_scaled[boot_rows])
    return boot_cols, model.coef_

def adaptive_feature_refinement(b, X_train_scaled, y_train_scaled, q2, p, importances, alpha_val):
    rs = np.random.default_rng(b + 200)
    n = X_train_scaled.shape[0]
    boot_rows = rs.choice(n, size=n, replace=True)
    
    prob_weights = importances / np.sum(importances)
    prob_weights = prob_weights / np.sum(prob_weights) 
    
    boot_cols = rs.choice(p, size=q2, replace=False, p=prob_weights)
    
    model = LassoLars(alpha=alpha_val, max_iter=2000)
    model.fit(X_train_scaled[boot_rows][:, boot_cols], y_train_scaled[boot_rows])
    return boot_cols, model.coef_



In [5]:
%%time
# ---------------------------------------------------------
# Main Execution (Self-Tuning Random LASSO)
# ---------------------------------------------------------
datasets = ["Dataset_I", "Dataset_II", "Dataset_III", "Dataset_IV"]
versions = ["v0", "v1", "v2", "v3", "v4", "v5", "v6", "v7", "v8", "v9"] 
#versions = ["v0"]
alphas = [10.0, 1.0, 1e-1, 1e-2, 1e-3]

rlasso_scores = []

for name in datasets:
    print(f"\nProcessing Random LASSO for {name}...")
    for v in versions:
        
        # 1. Load Data
        X = pd.read_csv(f'Simulation_Data/X_{name}_{v}.csv').values
        y = pd.read_csv(f'Simulation_Data/y_{name}_{v}.csv').values.ravel()
        true_betas = pd.read_csv(f'Simulation_Data/beta_{name}_{v}.csv').values.ravel()

        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=(1/6), random_state=42)
        X_scaler = StandardScaler()
        X_train_scaled = X_scaler.fit_transform(X_train)
        X_val_scaled = X_scaler.transform(X_val)
        
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_val_scaled = y_scaler.transform(y_val.reshape(-1, 1)).ravel()

        p = X.shape[1]
        true_selected = (true_betas != 0).astype(int)

        q1, q2 = X_train_scaled.shape[0], X_train_scaled.shape[0]
        B_rand = int((p / q1) * 30)

        best_r_rmse = np.inf
        best_r_f1, best_r_auprc, best_r_rme_all, best_r_rme_nz = 0, 0, 0, 0
        best_raw_coefs = None
        best_importances = None # Added to track Phase 1 scores

        # 2. FULL SWEEP
        for a in alphas:
            # --- INITIAL IMPORTANCE SCORING ---
            p1_results = Parallel(n_jobs=-1, batch_size=100)(
                delayed(generate_importance_scores)(b, X_train_scaled, y_train_scaled, q1, p, a) for b in range(B_rand)
            )
            
            p1_coef_sums = np.zeros(p)
            for boot_cols, boot_coefs in p1_results:
                p1_coef_sums[boot_cols] += boot_coefs
                
            # Random LASSO AUPRC Math: Average bootstraps first, then absolute value
            importances = np.abs(p1_coef_sums / B_rand)
            importances[importances == 0] = 1e-10
            
            # --- ADAPTIVE FEATURE REFINEMENT ---
            p2_results = Parallel(n_jobs=-1, batch_size=100)(
                delayed(adaptive_feature_refinement)(b, X_train_scaled, y_train_scaled, q2, p, importances, a) for b in range(B_rand)
            )
            
            rlasso_coef_sums = np.zeros(p)
            for boot_cols, boot_coefs in p2_results:
                rlasso_coef_sums[boot_cols] += boot_coefs
                
            raw_final_coef = rlasso_coef_sums / B_rand
            
            # --- HONEST ALPHA TUNING ---
            temp_coef_rmse = raw_final_coef.copy()
            temp_coef_rmse[np.abs(temp_coef_rmse) < 1e-6] = 0.0 
            
            r_pred = X_val_scaled @ temp_coef_rmse
            current_rmse = calc_rmse(y_val_scaled, r_pred)
            
            # The model is ONLY allowed to lock in its alpha based on Validation Error
            if current_rmse < best_r_rmse:
                best_r_rmse = current_rmse
                best_alpha = a
                best_raw_coefs = raw_final_coef.copy()
                best_importances = importances.copy() # Store specific importance scores for AUPRC

        # --- EVALUATION ---
        # The AUPRC is strictly evaluated using the Importance Scores as requested
        best_r_auprc = average_precision_score(true_selected, best_importances)
        
        # Validation RMSE Threshold Grid (Fair Selection)
        relative_fractions = [0.01, 0.025, 0.05, 0.10, 0.15, 0.20, 0.30]
        max_coef = np.max(np.abs(best_raw_coefs))
        
        best_r_rmse_eval = np.inf
        best_r_f1 = 0
        best_thresh = 0
        
        for frac in relative_fractions:
            thresh = frac * max_coef
            
            temp_eval = best_raw_coefs.copy()
            temp_eval[np.abs(temp_eval) < thresh] = 0.0
            
            r_pred = X_val_scaled @ temp_eval
            current_rmse = calc_rmse(y_val_scaled, r_pred)
            
            # HONEST SELECTION: pick the relative cutoff that minimizes Validation Error
            if current_rmse < best_r_rmse_eval:
                best_r_rmse_eval = current_rmse
                best_thresh = thresh
                
                best_r_f1 = f1_score(true_selected, (np.abs(temp_eval) > 0).astype(int))
                best_r_rme_all = calc_rme(true_betas, temp_eval)
                best_r_rme_nz = calc_rme_nonzeros(true_betas, temp_eval)

        best_r_rmse = best_r_rmse_eval

        rlasso_scores.append({
            "Dataset": name, "Seed": v, 
            "R_RMSE": best_r_rmse, "R_F1": best_r_f1, "R_AUPRC": best_r_auprc,
            "R_RME_All": best_r_rme_all, "R_RME_Nz": best_r_rme_nz
        })
        
        # --- NEW: Save Individual Coefficient CSV (Vertical) ---
        coef_df = pd.DataFrame({
            "Feature": [f"Beta_{i}" for i in range(p)],
            "Coefficient": best_raw_coefs
        })
        # Saves directly into the Random LASSO folder
        coef_df.to_csv(f"RandomLASSO_Coefs/{name}_{v}_RandomLASSO_coefs.csv", index=False)
        # -------------------------------------------------------
        
        print(f"  > Completed {v} | Alpha: {best_alpha} | Thresh: {best_thresh:.4f} | F1: {best_r_f1:.4f} | RMSE: {best_r_rmse:.4f}")

# ---------------------------------------------------------
# Export & Table Formatting
# ---------------------------------------------------------
results_df = pd.DataFrame(rlasso_scores)
results_df.to_csv("master_rlasso_metrics.csv", index=False)
print("\nDone! Saved peak scores to master_rlasso_metrics.csv.")

print("\n" + "="*90)
print("OPTIMAL RANDOM LASSO METRICS (± STD DEV)")
print("="*90)

summary_data = []
# Since the CSV only saved the best RMSE row per seed, we just group by Dataset
for base_name, group in results_df.groupby('Dataset'):
    row_dict = {"Dataset": base_name}
    
    # Calculate averages for EVERY metric
    for col in ["R_RMSE", "R_F1", "R_AUPRC", "R_RME_All", "R_RME_Nz"]: 
        mean_val = group[col].mean()
        std_val = group[col].std()
        
        std_str = f"{std_val:.4f}" if pd.notna(std_val) else "0.0000"
        row_dict[col] = f"{mean_val:.4f} ± {std_str}"
            
    summary_data.append(row_dict)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))


Processing Random LASSO for Dataset_I...
  > Completed v0 | Alpha: 0.001 | Thresh: 0.0648 | F1: 0.6667 | RMSE: 0.1886
  > Completed v1 | Alpha: 0.01 | Thresh: 0.0086 | F1: 0.4516 | RMSE: 0.3083
  > Completed v2 | Alpha: 0.1 | Thresh: 0.0040 | F1: 0.5714 | RMSE: 0.3197
  > Completed v3 | Alpha: 0.001 | Thresh: 0.0348 | F1: 0.5000 | RMSE: 0.2879
  > Completed v4 | Alpha: 0.01 | Thresh: 0.0662 | F1: 0.6667 | RMSE: 0.3022
  > Completed v5 | Alpha: 0.1 | Thresh: 0.0030 | F1: 0.7778 | RMSE: 0.2454
  > Completed v6 | Alpha: 0.01 | Thresh: 0.0625 | F1: 0.6667 | RMSE: 0.2045
  > Completed v7 | Alpha: 0.001 | Thresh: 0.0206 | F1: 0.5263 | RMSE: 0.2507
  > Completed v8 | Alpha: 0.01 | Thresh: 0.0371 | F1: 0.5882 | RMSE: 0.1189
  > Completed v9 | Alpha: 0.001 | Thresh: 0.0038 | F1: 0.2133 | RMSE: 0.2064

Processing Random LASSO for Dataset_II...
  > Completed v0 | Alpha: 0.01 | Thresh: 0.0026 | F1: 0.3663 | RMSE: 0.3641
  > Completed v1 | Alpha: 0.01 | Thresh: 0.0086 | F1: 0.7184 | RMSE: 0.2790
 